In [ ]:
import pandas as pd

# Load houseinfo.txt
column_names = ['bedrooms','bathrooms','area','zipcode','price']
df = pd.read_csv('/content/Houses-dataset-master/Houses_Dataset/HousesInfo.txt', sep=' ', names=column_names)

print(df.head())

   bedrooms  bathrooms  area  zipcode   price
0         4        4.0  4053    85255  869500
1         4        3.0  3343    36372  865200
2         3        4.0  3923    85266  889000
3         5        5.0  4022    85262  910000
4         3        4.0  4116    85266  971226


In [ ]:
import os

image_dir = 'images/'  # folder with all 2140 images
df['image_file'] = df.index + 1   # assuming index 0 = house 1
df['image_file'] = df['image_file'].apply(lambda x: f"{x}_1.jpg")  # frontal image

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# ---------------- Hyperparameters ----------------
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 20
LR = 1e-4
IMAGE_DIR = '/content/Houses-dataset-master/Houses_Dataset/'

# ---------------- Load Dataset ----------------
column_names = ['bedrooms', 'bathrooms', 'area', 'zipcode', 'price']
df = pd.read_csv(
    '/content/Houses-dataset-master/Houses_Dataset/HousesInfo.txt',
    sep=r'\s+',
    names=column_names,
    engine='python'
)

# Ensure numeric tabular columns
tab_features = ['bedrooms', 'bathrooms', 'area']
for col in tab_features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna().reset_index(drop=True)

# Map **frontal images** for each house
# Assuming filenames like "533_front.jpg"
df['image_file'] = df.index + 1
df['image_file'] = df['image_file'].apply(lambda x: f"{x}_frontal.jpg")

# Standardize tabular features
scaler = StandardScaler()
df[tab_features] = scaler.fit_transform(df[tab_features])

# Train / Validation split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

# ---------------- Dataset Class ----------------
class HouseDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Tabular data as float tensor
        tab_data = torch.tensor(row[tab_features].to_numpy(dtype=np.float32), dtype=torch.float32)

        # Load image
        img_path = os.path.join(self.img_dir, row['image_file'])
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        # Price target
        price = torch.tensor(row['price'], dtype=torch.float32)
        return tab_data, image, price

# ---------------- Transforms ----------------
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

train_ds = HouseDataset(train_df, IMAGE_DIR, transform)
val_ds = HouseDataset(val_df, IMAGE_DIR, transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)

# ---------------- Multimodal Model ----------------
class MultimodalNet(nn.Module):
    def __init__(self):
        super().__init__()
        # CNN feature extractor
        self.cnn = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        self.cnn = nn.Sequential(*list(self.cnn.children())[:-1])  # remove classifier

        # Fully connected layers for tabular + image features
        self.fc = nn.Sequential(
            nn.Linear(512 + len(tab_features), 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x_tab, x_img):
        img_feat = self.cnn(x_img)
        img_feat = img_feat.view(img_feat.size(0), -1)
        combined = torch.cat((x_tab, img_feat), dim=1)
        out = self.fc(combined)
        return out.squeeze()

# ---------------- Setup ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalNet().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# ---------------- Training Loop ----------------
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for tab, img, price in train_loader:
        tab, img, price = tab.to(device), img.to(device), price.to(device)
        optimizer.zero_grad()
        outputs = model(tab, img)
        loss = criterion(outputs, price)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    # Validation
    val_mae = 0
    val_rmse = 0
    model.eval()
    with torch.no_grad():
        for tab, img, price in val_loader:
            tab, img, price = tab.to(device), img.to(device), price.to(device)
            preds = model(tab, img)
            val_mae += torch.mean(torch.abs(preds - price)).item()
            val_rmse += torch.sqrt(torch.mean((preds - price) ** 2)).item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}, "
          f"Val MAE: {val_mae/len(val_loader):.4f}, Val RMSE: {val_rmse/len(val_loader):.4f}")

print("Training Complete!")

Epoch 1/20, Loss: 644678119272.2963, Val MAE: 545525.9241, Val RMSE: 651257.2946
Epoch 2/20, Loss: 649783814826.6666, Val MAE: 545520.4911, Val RMSE: 651250.8661
Epoch 3/20, Loss: 644599991258.0741, Val MAE: 545507.6741, Val RMSE: 651236.9241
Epoch 4/20, Loss: 646665763043.5555, Val MAE: 545507.4241, Val RMSE: 651234.6250
Epoch 5/20, Loss: 645585898458.0741, Val MAE: 545407.4420, Val RMSE: 651130.8661
Epoch 6/20, Loss: 646149558423.7037, Val MAE: 545436.8973, Val RMSE: 651149.5045
Epoch 7/20, Loss: 645063558940.4445, Val MAE: 545403.3259, Val RMSE: 651108.5625
Epoch 8/20, Loss: 643681251176.2963, Val MAE: 545366.7098, Val RMSE: 651056.3973
Epoch 9/20, Loss: 644499640471.7037, Val MAE: 545335.5804, Val RMSE: 651027.8571
Epoch 10/20, Loss: 646850860373.3334, Val MAE: 545207.8705, Val RMSE: 650862.1161
Epoch 11/20, Loss: 644869228923.2593, Val MAE: 545003.6339, Val RMSE: 650647.1875
Epoch 12/20, Loss: 647810385692.4445, Val MAE: 545322.0045, Val RMSE: 650987.9821
Epoch 13/20, Loss: 643794

In [ ]:
# ---------------- Save Model ----------------
MODEL_PATH = 'multimodal_model.pth'
SCALER_PATH = 'tabular_scaler.pkl'

# Save model state dict
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

# Save the tabular scaler (needed for preprocessing new data)
import joblib
joblib.dump(scaler, SCALER_PATH)
print(f"Tabular scaler saved to {SCALER_PATH}")

Model saved to multimodal_model.pth
Tabular scaler saved to tabular_scaler.pkl


In [ ]:
# 1️⃣ Load the scaler
import joblib
scaler = joblib.load('tabular_scaler.pkl')

# 2️⃣ Recreate the model and load weights
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultimodalNet().to(device)
model.load_state_dict(torch.load('multimodal_model.pth', map_location=device))
model.eval()  # Set to evaluation mode